In [1]:
import glob
import json
import os
import sys

import h5py
import lightning.pytorch as pl
import torch
import torch.nn.functional as F
import wandb
from lightning.fabric import Fabric
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.callbacks.model_checkpoint import ModelCheckpoint
from torch.nn import Embedding, Linear
from torch_geometric.data import Data, Dataset, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, radius_graph
from wandb.integration.lightning.fabric import WandbLogger

wandb: WARNING This integration is tested and supported for lightning Fabric 2.1.3.
wandb: WARNING             Please report any issues to https://github.com/wandb/wandb/issues with the tag `lightning-fabric`.


In [2]:
os.environ["WANDB_API_KEY"] = (
    "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"
)
wandb.login()

torch.set_float32_matmul_precision("high")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: clintvanhoesel (clintvanhoesel-eindhoven-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
if sys.version_info < (3,):
    RANGE_TYPE = list
else:
    RANGE_TYPE = range


def load(chkfile, key):
    """Load array(s) from chkfile

    Args:
        chkfile : str
            Name of chkfile. The chkfile needs to be saved in HDF5 format.
        key : str
            HDF5.dataset name or group name.  If key is the name of a HDF5
            group, the group will be loaded into a Python dict, recursively.

    Returns:
        whatever read from chkfile

    Examples:

    >>> from pyscf import gto, scf, lib
    >>> mol = gto.M(atom='He 0 0 0')
    >>> mf = scf.RHF(mol)
    >>> mf.chkfile = 'He.chk'
    >>> mf.kernel()
    >>> mo_coeff = lib.chkfile.load('He.chk', 'scf/mo_coeff')
    >>> mo_coeff.shape
    (1, 1)
    >>> scfdat = lib.chkfile.load('He.chk', 'scf')
    >>> scfdat.keys()
    ['e_tot', 'mo_occ', 'mo_energy', 'mo_coeff']
    """

    def load_as_dic(key, group):
        if key in group:
            val = group[key]
        elif key + "__from_list__" in group:
            key = key + "__from_list__"
            val = group[key]
        else:
            return None

        if isinstance(val, h5py.Group):
            if key.endswith("__from_list__"):
                return [load_as_dic(k, val) for k in val]
            else:
                return {
                    k.replace("__from_list__", ""): load_as_dic(k, val) for k in val
                }
        else:
            return val[()]

    with h5py.File(chkfile, "r") as fh5:
        return load_as_dic(key, fh5)


load_chkfile_key = load

In [4]:
def radius_graph_pbc(pos, r, box_size):
    """
    pos: (N, 3) positions inside [0, L)
    r: radius
    box_size: tensor/list of shape (3,) with box lengths
    """
    N = pos.size(0)
    device = pos.device

    # 1. Create periodic shifts: [-1, 0, 1] in each dimension
    shifts = torch.stack(
        torch.meshgrid(
            torch.tensor([-1, 0, 1], device=device),
            torch.tensor([-1, 0, 1], device=device),
            torch.tensor([-1, 0, 1], device=device),
            indexing="ij",
        ),
        dim=-1,
    ).reshape(
        -1, 3
    )  # (27, 3)

    # 2. Tile positions
    pos_images = pos.unsqueeze(0) + shifts.unsqueeze(1) * box_size
    pos_images = pos_images.reshape(-1, 3)  # (27*N, 3)

    # 3. Build graph on expanded positions
    edge_index = radius_graph(pos_images, r, loop=False)

    # 4. Keep only edges where the *source* is in the central cell
    mask = edge_index[0] < N
    edge_index = edge_index[:, mask]

    # 5. Map target indices back to primary cell
    target = edge_index[1] % N
    edge_index = torch.stack([edge_index[0], target], dim=0)

    return edge_index

In [5]:
def make_graph(pos, atom_types, radius):
    """
    pos: (N_atoms, 3)
    atom_types: (N_atoms,)
    """
    if not isinstance(pos, torch.Tensor):
        pos = torch.tensor(pos, dtype=torch.float)
    if not isinstance(atom_types, torch.Tensor):
        atom_types = torch.tensor(atom_types, dtype=torch.long).unsqueeze(-1)

    edge_index = radius_graph(
        pos, r=radius, loop=False, max_num_neighbors=64  # optional
    )

    return Data(x=atom_types, pos=pos, edge_index=edge_index)

In [6]:
class MolecularDataset(InMemoryDataset):
    def __init__(self, positions, atom_types, radius):
        super().__init__()
        self.positions = positions
        self.atom_types = atom_types
        self.radius = radius

    def len(self):
        return len(self.positions)

    def get(self, idx):
        pos = self.positions[idx]
        types = self.atom_types[idx]
        return make_graph(pos, types, self.radius)

In [7]:
class HDF_molecular_Dataset(MolecularDataset):
    def __init__(self, filepath, target, mol=None, radius=6.0):
        self.fp = filepath
        self.target = target
        self.mol = mol
        self.radius_val = radius

        # Load the data immediately upon initialization
        self.load_file()

    def load_molname_from_file(self, f=None):
        if f is None:
            with h5py.File(self.fp, "r") as f:
                key_list = list(f.keys())
                assert len(key_list) == 1
                self.mol = key_list[0]
        else:
            key_list = list(f.keys())
            assert len(key_list) == 1
            self.mol = key_list[0]

    def load_file(self):
        with h5py.File(self.fp, "r") as f:
            if self.mol is None:
                self.load_molname_from_file(f)

            group = f[self.mol]

            pos = torch.tensor(group["pos"][:], dtype=torch.float)

            atom_types = torch.tensor(group["types"][:], dtype=torch.long)

            if self.target in group:
                self.y = torch.tensor(group[self.target][:], dtype=torch.float)
            else:
                raise KeyError(
                    f"Target '{self.target}' not found in HDF5 group '{self.mol}'"
                )

        super().__init__(pos, atom_types, self.radius_val)

    def get(self, idx):
        # 1. Get the base graph structure from parent (pos + types -> graph)
        data = super().get(idx)

        # 2. Attach the target label to the graph object
        # self.y[idx] gives a scalar/vector. We usually keep dimensions consistent.
        data.y = self.y[idx].view(1, -1)

        # data.x = data.x.view(1, -1)

        return data

In [8]:
class GATMoleculeModel(torch.nn.Module):
    def __init__(self, num_atom_types, hidden_dim, num_heads=3, dropout=0.2):
        super().__init__()

        # 1. Embedding: Convert atom type integers (e.g., 1, 6, 8) to vectors
        # Assuming max atom type index is roughly 100 (Periodic table size)
        self.embedding = Embedding(num_atom_types, hidden_dim)

        # 2. GAT Layers
        # GATConv(in_channels, out_channels, heads, ...)
        # We concat heads, so output dim becomes hidden_dim * heads
        self.gat1 = GATConv(hidden_dim, hidden_dim, heads=num_heads, concat=True)
        self.gat2 = GATConv(
            hidden_dim * num_heads, hidden_dim, heads=num_heads, concat=True
        )

        # 3. Output prediction layer
        # Input is (hidden_dim * num_heads) because of the previous concatenation
        self.lin = Linear(hidden_dim * num_heads, 1)

        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        # x: Node features [num_nodes, 1] (atom types)
        # edge_index: Graph connectivity [2, num_edges]
        # batch: Batch vector mapping nodes to molecules [num_nodes]

        # 1. Embed atom types
        x = self.embedding(x).squeeze()  # [num_nodes, hidden_dim]

        # 2. GAT Layer 1
        x = self.gat1(x, edge_index)
        x = F.mish(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # 3. GAT Layer 2
        x = self.gat2(x, edge_index)
        x = F.mish(x)

        # 4. Global Pooling (aggregating node features to molecule features)
        # Takes the average of all node vectors belonging to the same graph
        x = global_mean_pool(x, batch)  # [batch_size, hidden_dim * heads]

        # 5. Final prediction
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin(x)

        return x

In [9]:
class LightningMoleculeModule(pl.LightningModule):
    def __init__(self, model, lr=0.001):
        super().__init__()
        self.model = model
        self.lr = lr
        self.save_hyperparameters(ignore=["model"])  # Saves args for checkpoints

    def forward(self, data):
        # Unpack PyG data object
        # data.x usually contains atom types, data.batch contains batch indices
        return self.model(data.x, data.edge_index, data.batch)

    def training_step(self, batch, batch_idx):
        # 1. Forward pass
        y_hat = self(batch)

        # 2. Calculate Loss (MSE for regression tasks like HOMO/LUMO)
        # Ensure shapes match: y is [batch_size, 1]
        loss = F.mse_loss(y_hat, batch.y)

        # 3. Log
        self.log(
            "train_loss",
            loss,
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            batch_size=batch.num_graphs,
        )
        return loss

    def validation_step(self, batch, batch_idx):
        y_hat = self(batch)
        val_loss = F.mse_loss(y_hat, batch.y)
        self.log("val_loss", val_loss, prog_bar=True, batch_size=batch.num_graphs)
        return val_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

In [10]:
HDF_FOLDER = "/home/clint/MultiLayerGNN/data"
hdf5_files = glob.glob(f"{HDF_FOLDER}/*.hdf5", recursive=False)
hdf5_files

['/home/clint/MultiLayerGNN/data/TAPC_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/alpha-NPB_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/MTDATA_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/beta-NPB_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/alpha-NPB_backup.hdf5',
 '/home/clint/MultiLayerGNN/data/alpha-MADN_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/Spiro-TAD_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/T2T_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/beta-NPB-2Me_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/BCP.hdf5',
 '/home/clint/MultiLayerGNN/data/alpha-NPB-2Me_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/mCP_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/BCP_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/mer-Alq3_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/2-TNATA_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/alpha-NPB.hdf5',
 '/home/clint/MultiLayerGNN/data/TCTA_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/NBPhen_ams.hdf5',
 '/home/clint/MultiLayerGNN/data/2-TNATA.hdf5']

In [11]:
HDF5_FILE = hdf5_files[0]
print(HDF5_FILE)
mol = HDF5_FILE.split(r"/")[-1].split(r".")[0].split(r"_")[0]
with h5py.File(HDF5_FILE) as f:
    f.visititems(print)

/home/clint/MultiLayerGNN/data/TAPC_ams.hdf5
TAPC <HDF5 group "/TAPC" (7 members)>
TAPC/HOMO <HDF5 dataset "HOMO": shape (396, 1), type "<f8">
TAPC/LUMO <HDF5 dataset "LUMO": shape (396, 1), type "<f8">
TAPC/Negative VIP <HDF5 dataset "Negative VIP": shape (396, 1), type "<f8">
TAPC/Positive VIP <HDF5 dataset "Positive VIP": shape (396, 1), type "<f8">
TAPC/lattice <HDF5 dataset "lattice": shape (3, 3), type "<f8">
TAPC/pos <HDF5 dataset "pos": shape (396, 94, 3), type "<f8">
TAPC/types <HDF5 dataset "types": shape (396, 94), type "<i8">


In [12]:
dataset = HDF_molecular_Dataset(HDF5_FILE, "Positive VIP", radius=2.0)

In [13]:
# 2. Split Data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

# 3. Create DataLoaders
# PyG DataLoaders handle collating graphs into batches automatically
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_set, batch_size=32, num_workers=4)

# 4. Initialize Model
# num_atom_types=100 covers most standard elements
gat_net = GATMoleculeModel(num_atom_types=20, hidden_dim=128, num_heads=8)

# 5. Initialize Lightning System
system = LightningMoleculeModule(model=gat_net, lr=1e-4)

callbacks = [EarlyStopping(monitor="val_loss", mode="min", patience=30)]

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    filename="best-{epoch}-{val_loss:.4f}"
)

callbacks.append(checkpoint_callback)


logger = WandbLogger(project="InitialGNNtrial")

fabric = Fabric(loggers=[logger])

# 6. Trainer
trainer = pl.Trainer(
    max_epochs=300,
    accelerator="auto",  # Automatically uses GPU if available
    devices=1,
    log_every_n_steps=10,
    callbacks=callbacks,
    logger=logger
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [14]:
trainer.fit(system, train_loader, val_loader)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type             | Params | Mode 
---------------------------------------------------
0 | model | GATMoleculeModel | 1.2 M  | train
---------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.758     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]